In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

c:\Users\Playdata\Desktop\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.


In [3]:
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print(type(model).__name__)

ChatOpenAI


In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 메이플스토리 정보 안내 챗봇입니다.

반드시 제공된 Context를 기반으로 답변하세요.
Context에 없는 정보는 임의로 만들어내지 마세요.
""".strip(),
        ),
        (
            "human",
            """
[Context]
{context}

[Question]
{question}
""".strip(),
        ),
    ]
)

In [ ]:
retrieve_runnable = RunnableLambda(lambda q : retrieve(question))

rag_chain = (
    {
        "context": retrieve_runnable | RunnableLambda(build_context), "question" : RunnablePassthrough()
    } | prompt | model | StrOutputParser()
)

def rag_answer(question):
    return rag_chain.invoke(question)